In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

pd.set_option("display.max_columns", None)

In [2]:
DATA_PATH = "../data/raw/slack_queries.csv"
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(481, 8)


,Ticket Id,Student or WP,Program Name,Status (Ticket),Created Time (Ticket),Ticket Closed Time,First Response Time,Project Phase
0,6403.0,Working Professionals,Fullstack Program,Closed,14-05-2021 01:09,14-05-2021 19:04,14-05-2021 08:51,trial phase
1,6415.0,Working Professionals,Backend Program,Duplicate,14-05-2021 10:12,14-05-2021 11:23,NaN,trial phase
2,6420.0,Working Professionals,Fullstack Program,Closed,14-05-2021 11:46,16-05-2021 00:09,14-05-2021 18:14,trial phase
3,6402.0,Student,Fullstack Program,Closed,14-05-2021 01:08,14-05-2021 19:04,14-05-2021 14:45,fullstack-phase-1
4,6423.0,Working Professionals,Fellowship Program,Closed,14-05-2021 12:17,14-05-2021 20:39,14-05-2021 12:21,trial phase


In [3]:
def analyze_missing_values(df):
    """
    Analyze missing values before imputation.

    Parameters
    ----------
    df : pandas DataFrame

    Returns
    -------
    DataFrame containing null counts and percentages.
    """

    analysis = pd.DataFrame({

        "Column": df.columns,

        "Null Count": df.isnull().sum().values,

        "Null Percentage":
            round(df.isnull().mean()*100,2).values,

        "Data Type":
            df.dtypes.values

    })

    print("="*70)
    print("Missing Value Analysis")
    print("="*70)

    display(analysis)

    print()

    print("Total Rows :", len(df))
    print("Total Missing :", df.isnull().sum().sum())

    return analysis

In [4]:
def impute_median(df, columns):

    df = df.copy()

    for col in columns:

        if col in df.columns:

            median = df[col].median()

            count = df[col].isna().sum()

            df[col].fillna(median, inplace=True)

            print(f"{col}: {count} values filled using median")

    return df

In [5]:
def impute_mode(df, columns):

    df = df.copy()

    for col in columns:

        if col in df.columns:

            mode = df[col].mode()[0]

            count = df[col].isna().sum()

            df[col].fillna(mode, inplace=True)

            print(f"{col}: {count} values filled using mode")

    return df

In [6]:
def forward_fill(df, columns):

    df = df.copy()

    for col in columns:

        if col in df.columns:

            count = df[col].isna().sum()

            df[col] = df[col].ffill()

            print(f"{col}: {count} values forward filled")

    return df

In [7]:
def drop_rows(df, columns):

    before = len(df)

    df = df.dropna(subset=columns)

    after = len(df)

    print(f"Rows Removed : {before-after}")

    return df

In [8]:
decisions = {

    "Ticket Id": {

        "strategy":"drop_rows",

        "reason":"Primary key cannot be missing."

    },

    "Status (Ticket)": {

        "strategy":"mode",

        "reason":"Status is categorical."

    },

    "First Response Time": {

        "strategy":"leave_null",

        "reason":"Duplicate or Deleted tickets legitimately have no first response."

    },

    "Program Name": {

        "strategy":"mode",

        "reason":"Business category."

    }

}

In [9]:
Path("../output").mkdir(exist_ok=True)

with open("../output/imputation_decisions.json","w") as f:

    json.dump(decisions,f,indent=4)

In [10]:
clean_df = df.copy()

clean_df = drop_rows(clean_df,["Ticket Id"])

clean_df = impute_mode(

    clean_df,

    [

        "Program Name",

        "Student or WP",

        "Project Phase",

        "Status (Ticket)"

    ]

)

Rows Removed : 5
Program Name: 0 values filled using mode
Student or WP: 0 values filled using mode
Project Phase: 0 values filled using mode
Status (Ticket): 0 values filled using mode


C:\Users\MOVVA KARTHIKA\AppData\Local\Temp\ipykernel_16308\1117291693.py:13: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].fillna(mode, inplace=True)


In [11]:
def validate(original, cleaned):

    print("="*70)

    print("Validation")

    print("="*70)

    print()

    print("Rows Before :",len(original))

    print("Rows After :",len(cleaned))

    print()

    print("Null Before :",original.isnull().sum().sum())

    print("Null After :",cleaned.isnull().sum().sum())

    print()

    display(pd.DataFrame({

        "Before":original.isnull().sum(),

        "After":cleaned.isnull().sum()

    }))

In [12]:
Path("../data/processed").mkdir(exist_ok=True)

clean_df.to_csv(

    "../data/processed/slack_queries_cleaned.csv",

    index=False

)

print("Dataset Saved Successfully")

Dataset Saved Successfully


In [13]:
print("Missing Value Handling Completed")

print(f"Rows : {len(clean_df)}")

print(f"Columns : {len(clean_df.columns)}")

print(f"Remaining Missing : {clean_df.isnull().sum().sum()}")

Missing Value Handling Completed
Rows : 476
Columns : 8
Remaining Missing : 110
